---
title: Slotted Union Finds
---

Co-written with Rudi Schneider

E-graphs are data structure for storing many equations and present a solution to the phase ordering problem in compilers. Slotted e-graphs are a proposal for compacting alpha equivalent terms in an e-graph. Many modelling use cases want summation $\Sigma$ or lambda terms and alpha equivalence is part of the story there.

I (Philip) have struggled to understand what is going on in slotted. I've sort of felt they don't fit into my way of thinking about things.

A general idea pump in e-graphs is

- hashcons + unionfind = egraph
- egraph - hashcons = union find
- egraph - union find = hash cons

It can be simpler to consider removing pieces from an egraph and look at what kind of hash cons or union find you get. Likewise exotic hash conses or union find lead to exotic e-graphs.

This blog post is about the slotted union find. In turn, we agree that it is kind of a nice abstraction for construction slotted e-graphs



# Basic Union Find
A union find is a forest where the pointers point up to their parents. This makes it easy to merge trees without eagerly updating all the children.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class UF():
    parents : list[int] = field(default_factory=list)
    def makeset(self):
        a = len(self.parents)
        self.parents.append(a)
        return a
    def find(self, a):
        while self.parents[a] != a:
            a = self.find(self.parents[a])
        return a
    def union(self, a, b):
        a,b = self.find(a), self.find(b)
        if a != b:
            self.parents[a] = b

    

# Group Union Finds

It is possible to add annotations to union finds. This gives them a bit more expressive power. They can be quite cheap sometimes. This may be useful in some applications.

Annotations could appear
- at roots
- at ids
- on edges

A noted form of this is the ability to add group elements to edges. The multiplication of the group is used to accumulate the annotation as you traverse up the tree in `find`. The identity is useful to give a null annotation. Inverse is useful in `union` when you need to find a way to isolate a annotated identifier to a bare identifier `ma * a = mb * b ===> a = ma^-1 * mb * b`  


Here is probably the simplest version of this, using the integers `Z` considered as an offset group


In [ ]:
type Offset = int
type Id = int
type OId = tuple[Offset, Id]
class OffsetUF():
    parents : list[OId] = field(default_factory=list)
    def makeset(self) -> OId:
        a = len(self.parents)
        oid = (0,a)
        self.parents.append(oid)
        return oid
    def find(self, oid: OId) -> OId:
        (off, a) = oid
        while True:
            (off1, b) = self.parents[a]
            if b == a:
                assert off1 == 0
                return (off, a)
            off += off1
            a = b 
    def union(self, a: OId, b: OId):
        (offa, ida) = a
        (offb, idb) = b
        ida, idb = self.find((offa, ida))[1], self.find((offb, idb))[1]
        if ida != idb:
            self.parents[ida] = (offa - offb, idb)
        elif ida == idb:
            if offa != offb:
                raise Exception("offset mismatch")



# Permutation Groups

In an e-graph, one of the interpretations is that eclasses represent a set of equivalent terms `{f(bar,biz), f(biz,baz)}. In the slotted e-graph, these terms contain special names that are intended to be alpha invariant  `{f(a, b), g(a,a,b)}`. This set is _related_ but _not equal_ to another set `{f(a1, b1), g(a1,a1,b1)}` by a renaming process of some kind. This is akin to how if we know that two things are offset related `x = y + 42` this does not mean they are necessarily _equal_. We want to factor out this renaming process so that we can compactly represent the many renaming related but nevertheless distinct sets of terms.

Bijective renamings can be thought of as permutations. Permutations form a group.



In [ ]:
class Perm():
    

# Symmetries



# Redundancies

# Bits and Bobbles

My blog posts
Rudi's
Slotted Paper and talks
github repo

group union find paper
ed kmett
groupoid blog post


